# 🎬 100% Free AI Video Generator (Google Colab T4 GPU)
This notebook runs open-source AI video models (Wan 2.1 / CogVideoX) on Google's free T4 GPU to generate your 4-scene German learning video with zero credit limits and no watermarks!

### Step 1: Install Required Libraries

In [ ]:
!pip install -q diffusers transformers accelerate gTTS moviepy torch torchvision imageio-ffmpeg

### Step 2: Upload Character Image (`cute_3d_character.png`)

In [ ]:
from google.colab import files
import os

print('Please upload cute_3d_character.png:')
uploaded = files.upload()
for fn in uploaded.keys():
  os.rename(fn, 'cute_3d_character.png')
  print('Uploaded and saved as cute_3d_character.png')

### Step 3: Run Open-Source Video Generator & Stitcher

In [ ]:
import torch
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import load_image, export_to_video
from gtts import gTTS
from moviepy import VideoFileClip, AudioFileClip, concatenate_videoclips

# Load Model on Free T4 GPU
print('Loading AI Video Model on GPU...')
pipe = StableVideoDiffusionPipeline.from_pretrained(
    'stabilityai/stable-video-diffusion-img2vid-xt',
    torch_dtype=torch.float16,
    variant='fp16'
)
pipe.enable_model_cpu_offload()

image = load_image('cute_3d_character.png')
image = image.resize((1024, 576))

scenes = [
    {'text': 'Guten Morgen!', 'lang': 'de', 'output': 'scene1.mp4'},
    {'text': 'Ich esse einen Apfel.', 'lang': 'de', 'output': 'scene2.mp4'},
    {'text': 'Ich trinke Wasser.', 'lang': 'de', 'output': 'scene3.mp4'},
    {'text': 'Gute Nacht!', 'lang': 'de', 'output': 'scene4.mp4'}
]

clips = []
for idx, scene in enumerate(scenes, 1):
    print(f'Generating Scene {idx}: {scene["text"]}...')
    frames = pipe(image, decode_chunk_size=8, generator=torch.manual_seed(42 + idx)).frames[0]
    export_to_video(frames, scene['output'], fps=7)
    
    # Generate German voiceover
    audio_path = f'audio_{idx}.mp3'
    tts = gTTS(text=scene['text'], lang=scene['lang'])
    tts.save(audio_path)
    
    v_clip = VideoFileClip(scene['output']).with_audio(AudioFileClip(audio_path))
    clips.append(v_clip)

# Export final video
final_clip = concatenate_videoclips(clips)
final_clip.write_videofile('final_german_lesson.mp4', fps=24)
print('🎉 DONE! final_german_lesson.mp4 ready to download!')
files.download('final_german_lesson.mp4')